# 배터리 정책 분석 — 균등성 / 5.3 표 / 병렬 GIF (Colab Runner)

배터리 인지 정책이 (1) 임무 성능, (2) **드론 간 배터리 잔량의 균등성**, (3) 개별 드론의
에너지 절약 행동에 미치는 영향을 분석한다.

- **세 정책**(비-점유, 같은 시퀀스/하이퍼파라미터로 학습):
  - **NB**: 배터리 없음 (`comm_train.py`, obs 71)
  - **B_tight**: 배터리 + 빠듯한 예산 (hover .002 / move .005 / penalty .15)
  - **B_relaxed**: 배터리 + 완화 예산 (hover .001 / move .0025 / penalty .05)
- **공정 비교 트릭**: 배터리 환경 관측은 `[기본 71] + [배터리 14]` 구조 → NB(71차원) 정책에는
  관측 앞 71차원만 잘라 넣어 **같은 배터리 환경**에서 동일하게 평가한다.

**런타임**: GPU 권장. 학습 3개는 시간이 걸리므로 이미 ckpt가 있으면 자동 skip.

## 1. GitHub clone

In [ ]:
BRANCH = "Saehoon"
%cd /content
!rm -rf /content/RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git /content/RL-2026s1-tp
%cd /content/RL-2026s1-tp
!git switch $BRANCH && git pull origin $BRANCH
!git log --oneline -1

## 2. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery_letters"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/gifs {DRIVE_ROOT}/evals {DRIVE_ROOT}/analysis
print("DRIVE_ROOT =", DRIVE_ROOT)

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. 설정

In [ ]:
TARGET_SEQUENCE = "GROUND,D,G"
EXP_TAG  = "DG"
GRID_SIZE, N_AGENTS, MAX_STEPS = 25, 14, 500
SHAPES = [s.strip() for s in TARGET_SEQUENCE.split(",") if s.strip()]

# 학습 하이퍼파라미터 (세 정책 공통)
TOTAL_FRAMES, FRAMES_PER_BATCH, MINIBATCH = 800_000, 4096, 512
PPO_EPOCHS, LR, ENT_COEF, CLIP_EPS = 6, 2e-4, 0.02, 0.15
CKPT_EVERY = 5
COMPLETION_REWARD, ASSIGNED, COVDELTA, COVSTEP, HOVERP, SHAPING = 120.0, 0.4, 0.3, 0.01, 0.05, 0.5
EARLY_STOP, EARLY_PAT, SAVE_BEST = 1.0, 3, 0.5

# 배터리 예산 프리셋
TIGHT   = dict(init=1.0, hover=0.002, move=0.005,  pen=0.15)
RELAXED = dict(init=1.0, hover=0.001, move=0.0025, pen=0.05)

NB_DIR = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_nb"            # 배터리 없음
BT_DIR = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_batt_tight"    # 배터리 tight
BR_DIR = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_batt_relaxed"  # 배터리 relaxed
print("NB/BT/BR:", NB_DIR, BT_DIR, BR_DIR, sep="\n  ")

## 5. ckpt 헬퍼

In [ ]:
import os, re, glob
def latest_ckpt(d):
    cand=[]
    for p in glob.glob(os.path.join(d,'ckpt_*.pt')):
        m=re.search(r'ckpt_(\d+)\.pt$',p)
        if m: cand.append((int(m.group(1)),p))
    if not cand: return None,0
    cand.sort(); return cand[-1][1],cand[-1][0]
def eval_ckpt(d):
    b=os.path.join(d,"ckpt_best.pt")
    return b if os.path.exists(b) else latest_ckpt(d)[0]

## 6. 세 정책 학습 (이미 있으면 skip)

NB는 `comm_train.py`(배터리 없음), B_tight/B_relaxed는 `comm_train_battery.py`. 모두 비-점유.

In [ ]:
import os
def run_if_needed(save_dir, cmd):
    if latest_ckpt(save_dir)[0]:
        print(f"[skip] 이미 학습됨: {latest_ckpt(save_dir)[0]}"); return
    os.makedirs(save_dir, exist_ok=True)
    print(cmd); get_ipython().system(cmd)

common = (f"--grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} "
          f"--shapes '{TARGET_SEQUENCE}' --completion-reward {COMPLETION_REWARD} "
          f"--assigned-target-reward {ASSIGNED} --coverage-delta-reward {COVDELTA} "
          f"--coverage-step-reward {COVSTEP} --hover-penalty {HOVERP} --shaping-coef {SHAPING} "
          f"--total-frames {TOTAL_FRAMES} --frames-per-batch {FRAMES_PER_BATCH} --minibatch-size {MINIBATCH} "
          f"--ppo-epochs {PPO_EPOCHS} --lr {LR} --ent-coef {ENT_COEF} --clip-eps {CLIP_EPS} "
          f"--ckpt-every {CKPT_EVERY} --early-stop-success {EARLY_STOP} --early-stop-patience {EARLY_PAT} "
          f"--save-best-above {SAVE_BEST}")

# NB (배터리 없음)
run_if_needed(NB_DIR, f"python comm_train.py {common} --save-dir {NB_DIR} --tb-logdir {DRIVE_ROOT}/runs/{EXP_TAG}_nb")
# B_tight
b=TIGHT
run_if_needed(BT_DIR, f"python comm_train_battery.py {common} --initial-battery {b['init']} "
              f"--hover-battery-cost {b['hover']} --move-battery-cost {b['move']} --low-battery-move-penalty {b['pen']} "
              f"--save-dir {BT_DIR} --tb-logdir {DRIVE_ROOT}/runs/{EXP_TAG}_batt_tight")
# B_relaxed
b=RELAXED
run_if_needed(BR_DIR, f"python comm_train_battery.py {common} --initial-battery {b['init']} "
              f"--hover-battery-cost {b['hover']} --move-battery-cost {b['move']} --low-battery-move-penalty {b['pen']} "
              f"--save-dir {BR_DIR} --tb-logdir {DRIVE_ROOT}/runs/{EXP_TAG}_batt_relaxed")

## 7. 공통 분석 헬퍼 (먼저 실행)

`run_in_batt(actor, obs_in, ...)`: **배터리 환경**에서 정책을 굴린다. NB(obs_in=71)는 관측 앞 71차원만
넣어 같은 환경에서 평가(배터리 회계 동일). 프레임·드론별 최종 배터리를 기록.

In [ ]:
import torch, numpy as np
from torchrl.envs.utils import ExplorationType, set_exploration_type, step_mdp
import comm_eval as ce
import comm_eval_battery as ceb
from comm_eval_battery import _battery_color as bcolor
GROUP = ceb.GROUP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_actor(ckpt, obs_dim, env):
    a = ceb.build_actor(obs_dim, 5, N_AGENTS, 128, device)
    with torch.no_grad(): a(env.reset())
    a.load_state_dict(torch.load(ckpt, map_location=device)["actor"]); a.eval()
    return a

def make_batt(seed=0, batt=RELAXED, comm=0.0, wind=0.0):
    return ceb.make_env(seed=seed, device=device, grid_size=GRID_SIZE, n_agents=N_AGENTS,
        max_steps=MAX_STEPS, shapes=SHAPES, comm_fail_prob=comm, completion_reward=COMPLETION_REWARD,
        wind_prob=wind, wind_strength=1, randomize_wind=False,
        initial_battery=batt['init'], hover_battery_cost=batt['hover'],
        move_battery_cost=batt['move'], low_battery_move_penalty=batt['pen'])

def run_in_batt(actor, obs_in, greedy=True, record=False, seed=0, batt=RELAXED):
    env, base = make_batt(seed=seed, batt=batt)
    td = env.reset()
    expl = ExplorationType.MODE if greedy else ExplorationType.RANDOM
    frames = []
    def snap():
        return dict(positions=dict(base.agent_pos), batteries=dict(base.battery),
                    target_cells=list(base.target_cells))
    if record: frames.append(snap())
    success = False
    for _ in range(base.max_steps):
        full = td.get((GROUP, "observation"))
        sliced = full.shape[-1] != obs_in
        if sliced: td.set((GROUP, "observation"), full[..., :obs_in])
        with set_exploration_type(expl), torch.no_grad():
            actor(td)
        if sliced: td.set((GROUP, "observation"), full)
        td = env.step(td)
        if record: frames.append(snap())
        if bool(td.get(("next", GROUP, "done")).all().item()):
            success = bool(td.get(("next", GROUP, "terminated")).any().item()); break
        td = step_mdp(td)
    cov = getattr(base, "last_occupied_count", 0) / max(1, len(base.target_cells))
    return dict(success=success, coverage=cov, final_batt=dict(base.battery), frames=frames)

print("헬퍼 준비 완료: load_actor / make_batt / run_in_batt")

## 8. (5.3 표) 배터리 없음 / tight / relaxed 성능 비교

각 정책을 평가해 성공률(greedy/stochastic)·평균 커버리지·최종 배터리 평균을 표로.

In [ ]:
N_EVAL = 50

def eval_table_row(ckpt_dir, obs_in, batt, has_batt):
    ck = eval_ckpt(ckpt_dir)
    assert ck, f"{ckpt_dir}: ckpt 없음 (먼저 학습)"
    # actor 초기화용 임시 배터리 env
    env0,_ = make_batt(seed=0, batt=batt)
    actor = load_actor(ck, obs_in, env0)
    def rate(greedy):
        runs=[run_in_batt(actor, obs_in, greedy=greedy, seed=1000+i, batt=batt) for i in range(N_EVAL)]
        succ=np.mean([r["success"] for r in runs]); cov=np.mean([r["coverage"] for r in runs])
        fb=np.mean([np.mean(list(r["final_batt"].values())) for r in runs])
        return succ, cov, fb
    sg,covg,_  = rate(True)
    ss,covs,fb = rate(False)
    cov=(covg+covs)/2
    return dict(succ_g=sg, succ_s=ss, cov=cov, fb=(fb if has_batt else None))

rows = [
    ("배터리 없음(NB)", NB_DIR, 71, RELAXED, False),  # NB는 배터리 환경에서 굴리되 배터리는 무시(슬라이스)
    ("배터리 (tight)",  BT_DIR, 85, TIGHT,   True),
    ("배터리 (relaxed)",BR_DIR, 85, RELAXED, True),
]
out=[]
def emit(x): out.append(x); print(x)
emit(f"=== 5.3 성능 비교 (n={N_EVAL}, 시퀀스 {TARGET_SEQUENCE}) ===")
hdr="환경".ljust(18)+"succ(greedy)".rjust(13)+"succ(stoch)".rjust(13)+"coverage".rjust(11)+"final_batt".rjust(12)
emit(hdr); emit("-"*len(hdr))
for name,d,oi,bt,hb in rows:
    m=eval_table_row(d,oi,bt,hb)
    fb = f"{m['fb']*100:.1f}%" if m['fb'] is not None else "—"
    emit(name.ljust(18)+f"{m['succ_g']*100:.0f}%".rjust(13)+f"{m['succ_s']*100:.0f}%".rjust(13)
         +f"{m['cov']*100:.1f}%".rjust(11)+fb.rjust(12))
open(f"{DRIVE_ROOT}/analysis/table53_{EXP_TAG}.txt","w",encoding="utf-8").write("\n".join(out)+"\n")
print("saved ->", f"{DRIVE_ROOT}/analysis/table53_{EXP_TAG}.txt")

## 9. (균등성) NB 정책 vs 배터리 정책 — 드론별 최종 배터리 분포 + box plot

**같은 배터리 환경**에서 NB 정책(앞 71차원만 사용)과 배터리 정책(relaxed)을 각각 굴려,
드론별 최종 배터리 잔량을 모은다. 균등할수록 분산↓, 최소↑, Jain↑.

In [ ]:
import numpy as np, matplotlib
import matplotlib.pyplot as plt

N_FAIR = 40
nb_ck, b_ck = eval_ckpt(NB_DIR), eval_ckpt(BR_DIR)
env0,_=make_batt(); nb_actor=load_actor(nb_ck,71,env0)
env1,_=make_batt(); b_actor =load_actor(b_ck,85,env1)

def collect(actor, obs_in):
    vals=[]  # per-episode list of 14 final batteries
    for i in range(N_FAIR):
        r=run_in_batt(actor, obs_in, greedy=False, seed=2000+i, batt=RELAXED)
        vals.append(np.array(list(r["final_batt"].values())))
    return np.array(vals)  # (N_FAIR, 14)

NBv, Bv = collect(nb_actor,71), collect(b_actor,85)

def jain(x):  # per-episode Jain fairness, then mean
    s1=x.sum(axis=1); s2=(x**2).sum(axis=1)
    return float(np.mean(s1**2/(x.shape[1]*np.maximum(s2,1e-9))))
def stats(v):
    return dict(mean=v.mean(), std=float(np.mean(v.std(axis=1))),
                mn=float(np.mean(v.min(axis=1))), spread=float(np.mean(v.max(axis=1)-v.min(axis=1))),
                jain=jain(v))
sN, sB = stats(NBv), stats(Bv)
print(f"{'metric':<18}{'NB 정책':>12}{'배터리 정책':>14}")
for k,lab in [('mean','평균 잔량'),('std','표준편차↓'),('mn','최소 잔량↑'),('spread','max-min폭↓'),('jain','Jain↑')]:
    fb=lambda x:(f"{x*100:.1f}%" if k!='jain' else f"{x:.3f}")
    print(f"{lab:<18}{fb(sN[k]):>12}{fb(sB[k]):>14}")

# box plot: 드론별 최종 배터리 분포
plt.figure(figsize=(6,5))
plt.boxplot([NBv.flatten()*100, Bv.flatten()*100], labels=["NB policy","battery policy"], showmeans=True)
plt.ylabel("final battery (%)"); plt.title(f"Final battery distribution per drone  [{TARGET_SEQUENCE}]")
plt.grid(axis="y", alpha=0.3)
png=f"{DRIVE_ROOT}/analysis/fairness_box_{EXP_TAG}.png"
plt.tight_layout(); plt.savefig(png, dpi=120); plt.show()
print("saved ->", png)

## 10. (병렬 GIF) 잔량 급감 드론 비교

NB 정책 실행에서 **최종 잔량이 가장 낮은(가장 많이 소모한) 드론**을 하나 골라, NB와 배터리 정책의
에피소드를 **나란히** 보여준다. 해당 드론에 자홍색 링 + ★ 표식 → NB는 계속 활발히 움직여(잔량 급감),
배터리 정책은 점차 움직임을 줄임(잔량 완만)을 시각적으로 대비.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, PillowWriter

# 한 에피소드씩 기록 (stochastic, 동일 seed로 초기 배치 맞춤)
SEED = 7
nb_run = run_in_batt(nb_actor, 71, greedy=False, record=True, seed=SEED, batt=RELAXED)
b_run  = run_in_batt(b_actor,  85, greedy=False, record=True, seed=SEED, batt=RELAXED)

# NB에서 최종 잔량 최저 드론 선정
mark = min(nb_run["final_batt"], key=nb_run["final_batt"].get)
print("표식 드론:", mark, "| NB 최종 잔량:", f"{nb_run['final_batt'][mark]*100:.0f}%",
      "| batt 정책 최종 잔량:", f"{b_run['final_batt'][mark]*100:.0f}%")

def save_parallel(histA, histB, titleA, titleB, mark, path, fps=4):
    n=max(len(histA),len(histB))
    fig,axes=plt.subplots(1,2,figsize=(13,7),facecolor="#0a0a14")
    def one(ax,hist,step,title):
        f=hist[min(step,len(hist)-1)]; ax.clear(); ax.set_facecolor("#0a0a14")
        ax.set_xlim(-0.5,GRID_SIZE-0.5); ax.set_ylim(-0.5,GRID_SIZE-0.5); ax.invert_yaxis()
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        tset=set(f["target_cells"]); mb=f["batteries"].get(mark,0.0)
        ax.set_title(f"{title}\nstep {min(step,len(hist)-1)} | {mark} batt={mb*100:.0f}%", color="#ddd", fontsize=11)
        for (r,c) in f["target_cells"]:
            ax.add_patch(patches.Rectangle((c-0.5,r-0.5),1,1,facecolor="#1c1c2e",edgecolor="#3a3a55",lw=0.6,ls=(0,(2,2))))
        for ag,(r,c) in f["positions"].items():
            on=(r,c) in tset
            ax.add_patch(patches.Circle((c,r),0.34 if on else 0.22,
                facecolor="#ffd24a" if on else "#3a3a44", edgecolor="#fff4b3" if on else "#555", lw=1.2))
            b=f["batteries"].get(ag,0.0)
            ax.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9,0.13,facecolor="#222230",edgecolor="#777",lw=0.3))
            if b>0: ax.add_patch(patches.Rectangle((c-0.45,r-0.78),0.9*max(0,min(1,b)),0.13,facecolor=bcolor(b),edgecolor="none"))
            if ag==mark:
                ax.add_patch(patches.Circle((c,r),0.62,facecolor="none",edgecolor="#ff45ff",lw=2.6))
                ax.text(c,r+0.85,"★",color="#ff45ff",ha="center",va="top",fontsize=12,fontweight="bold")
    def draw(step): one(axes[0],histA,step,titleA); one(axes[1],histB,step,titleB)
    anim=FuncAnimation(fig,draw,frames=n,interval=1000//fps)
    anim.save(path,writer=PillowWriter(fps=fps)); plt.close(fig)

gif=f"{DRIVE_ROOT}/gifs/parallel_{EXP_TAG}_{mark}.gif"
save_parallel(nb_run["frames"], b_run["frames"], "NB policy (배터리 무시)", "battery policy (절약)", mark, gif)
print("saved ->", gif)
from IPython.display import Image, display
display(Image(gif))